# 11 — Khóa kết quả nghiên cứu chính thức (Gate G5)

**Mục tiêu:** Xác nhận các run hợp lệ, tạo checksum SHA256 cho toàn bộ bảng biểu, mô hình và khóa final_results_manifest.json.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chọn `runtime` để chạy không cần Drive, hoặc `drive` để 13 notebook dùng chung dữ liệu bền vững. Với `drive`, mọi notebook phải dùng cùng `PUBG_DRIVE_PROJECT_ROOT` và chạy theo thứ tự.


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "runtime"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Project_PUBG"  # @param {type:"string"}


In [ ]:
# Bootstrap: runtime mode needs no Drive; drive mode persists stage outputs.
import base64
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "runtime").strip().lower()
if PUBG_STORAGE_MODE not in {"runtime", "drive"}:
    raise ValueError("PUBG_STORAGE_MODE must be 'runtime' or 'drive'")

if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Drive mode is available only on Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(globals().get(
        "PUBG_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/Project_PUBG"
    )).expanduser().resolve()
else:
    _candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/Project_PUBG")]
    _candidates += [p / "Project_PUBG" for p in list(_candidates)]
    PROJECT_ROOT = next((p.resolve() for p in _candidates
                         if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"

if not (PROJECT_ROOT / "configs/data.yaml").is_file() or not (PROJECT_ROOT / "src/utils/config.py").is_file():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])
    _PUBG_PACKAGES_READY = True

from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
if PUBG_STORAGE_MODE == "drive":
    cfg["paths"]["environments"]["drive"] = {
        "raw_root": str(PROJECT_ROOT / "data/raw"),
        "data_root": str(PROJECT_ROOT / "data"),
        "artifacts_root": str(PROJECT_ROOT / "artifacts"),
        "figures_root": str(PROJECT_ROOT / "figures"),
        "reports_root": str(PROJECT_ROOT / "reports"),
        "temp_dir": globals().get("PUBG_RUNTIME_TEMP_DIR", "/content/temp"),
    }
    cfg["paths"]["active_environment"] = "drive"
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
if PUBG_STORAGE_MODE == "drive":
    print("Storage mode: Google Drive. Stage outputs persist for the next notebook.")
else:
    print("Storage mode: runtime. No Drive authorization required; export before reset.")


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.evaluation.finalize import build_final_results_manifest

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)

official_runs = {
    "rq1": "rq1_relationship_summary_v1",
    "p1": "p1_linear",
    "p2": "p2_linear",
    "ablation": "ablation_p2_groups",
}
import pandas as pd
cluster_table = paths["tables"] / "cluster_profile.csv"
if cluster_table.is_file():
    official_runs["rq2"] = f"rq2_kmeans_k{len(pd.read_csv(cluster_table))}"
for required in [paths["tables"] / "rq1_relationship_summary.csv",
                 paths["experiments"] / "predictions_p1_linear.parquet",
                 paths["experiments"] / "predictions_p2_linear.parquet",
                 paths["tables"] / "ablation_results.csv"]:
    if not required.is_file():
        raise FileNotFoundError(f"Chưa chạy xong các bước trước: {required}")

manifest = build_final_results_manifest(
    artifacts_root=paths["artifacts_root"],
    reports_root=paths["reports_root"],
    official_run_ids=official_runs,
    output_manifest_path=paths["manifests"] / "final_results_manifest.json"
)

print(f"Gate G5 Đã khóa: {len(manifest['tables'])} bảng kết quả chính thức đã được băm mã hóa bảo vệ.")